# 02 — Modélisation\n\nPipeline complet de modélisation :\n1. Chargement et prétraitement\n2. Baseline\n3. Régression Logistique\n4. Random Forest\n5. XGBoost\n6. Comparaison des modèles\n\n> ⚠️ L'attribut `duration` est **exclu** des modèles (car inconnu avant la fin de l'appel).

In [ ]:
# Imports\nimport pandas as pd\nimport numpy as np\nimport sys\nsys.path.insert(0, '..')\n\nfrom src.data_loader import load_bank, load_bank_additional\nfrom src.preprocessing import (\n    separate_features_target,\n    get_column_types,\n    split_data,\n    build_preprocessing_pipeline,\n)\nfrom src.modeling import (\n    get_baseline,\n    get_logistic_regression,\n    get_random_forest,\n    get_xgboost,\n    tune_model,\n)\nfrom src.evaluation import (\n    get_metrics,\n    print_classification_report,\n    plot_confusion_matrix,\n    plot_roc_curve,\n    plot_feature_importance,\n    compare_models,\n)\nfrom src.utils import set_seed, Timer, save_model\n\nset_seed(42)\n%matplotlib inline

## 1. Chargement des données

In [ ]:
# Choisir le dataset à utiliser\ndf = load_bank(full=True)          # Option 1 : Bank (UCI original)\n# df = load_bank_additional(full=True)  # Option 2 : Bank Additional (enrichi)\n\nprint(f"Shape: {df.shape}")

## 2. Séparation features / target

In [ ]:
X, y = separate_features_target(df, drop_duration=True)\nprint(f"X: {X.shape}, y: {y.shape}")\nprint(f"Class distribution:\\n{y.value_counts()}")

## 3. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, stratify=True)

## 4. Prétraitement

In [ ]:
num_cols, cat_cols = get_column_types(X_train)\nprint(f"Numerical columns ({len(num_cols)}): {num_cols}")\nprint(f"Categorical columns ({len(cat_cols)}): {cat_cols}")\n\npreprocessor = build_preprocessing_pipeline(num_cols, cat_cols)\nX_train_pp = preprocessor.fit_transform(X_train)\nX_test_pp = preprocessor.transform(X_test)\n\nprint(f"X_train after preprocessing: {X_train_pp.shape}")

## 5. Modèle baseline

In [ ]:
baseline = get_baseline(strategy='stratified')\nbaseline.fit(X_train_pp, y_train)\ny_pred_base = baseline.predict(X_test_pp)\nprint("Baseline (stratified):")\nprint(get_metrics(y_test, y_pred_base))

## 6. Régression Logistique

In [ ]:
lr = get_logistic_regression(class_weight='balanced', max_iter=2000)\nwith Timer("Logistic Regression"):\n    lr.fit(X_train_pp, y_train)\ny_pred_lr = lr.predict(X_test_pp)\ny_proba_lr = lr.predict_proba(X_test_pp)[:, 1]\nprint_classification_report(y_test, y_pred_lr)

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr, title='Logistic Regression')\nplot_roc_curve(y_test, y_proba_lr, label='Logistic Regression')

## 7. Random Forest

In [ ]:
rf = get_random_forest(n_estimators=200, class_weight='balanced')\nwith Timer("Random Forest"):\n    rf.fit(X_train_pp, y_train)\ny_pred_rf = rf.predict(X_test_pp)\ny_proba_rf = rf.predict_proba(X_test_pp)[:, 1]\nprint_classification_report(y_test, y_pred_rf)

In [ ]:
plot_confusion_matrix(y_test, y_pred_rf, title='Random Forest')\nplot_roc_curve(y_test, y_proba_rf, label='Random Forest')\nplot_feature_importance(rf, num_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)))

## 8. XGBoost

In [ ]:
neg, pos = np.bincount(y_train)\nscale_pos_weight = neg / pos\nxgb = get_xgboost(scale_pos_weight=scale_pos_weight, n_estimators=200)\nwith Timer("XGBoost"):\n    xgb.fit(X_train_pp, y_train)\ny_pred_xgb = xgb.predict(X_test_pp)\ny_proba_xgb = xgb.predict_proba(X_test_pp)[:, 1]\nprint_classification_report(y_test, y_pred_xgb)

In [ ]:
plot_confusion_matrix(y_test, y_pred_xgb, title='XGBoost')\nplot_roc_curve(y_test, y_proba_xgb, label='XGBoost')\nplot_feature_importance(xgb, num_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)))

## 9. Comparaison des modèles

In [ ]:
results = {\n    "Baseline": get_metrics(y_test, y_pred_base),\n    "Logistic Regression": get_metrics(y_test, y_pred_lr, y_proba_lr),\n    "Random Forest": get_metrics(y_test, y_pred_rf, y_proba_rf),\n    "XGBoost": get_metrics(y_test, y_pred_xgb, y_proba_xgb),\n}\ncompare_models(results, sort_by="ROC-AUC")

## 10. Sauvegarde du meilleur modèle

In [ ]:
# save_model(rf, 'random_forest_bank')